###  Step 1: Install Required Packages

In [46]:
!pip install faiss-cpu

In [47]:
!pip install langchain langchain-community langchain-groq python-dotenv

###  Step 2: Import Modules & Set Logging

In [48]:
import os
import pickle
import logging
from dotenv import load_dotenv
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_groq import ChatGroq
from langchain.chains import RetrievalQA

# Set up logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

###  Step 3: Load Environment Variables

##### ➡️ Make sure your .env file contains:

In [49]:
GROQ_API_KEY="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"

In [50]:
# from dotenv import load_dotenv
# import os

# load_dotenv()  # Loads .env file into environment

# # Check if the variable is available
# print("GROQ_API_KEY:", os.getenv("GROQ_API_KEY"))


In [59]:
# Load API key
# load_dotenv()
# GROQ_API_KEY = os.getenv("GROQ_API_KEY")

GROQ_API_KEY = "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"

if not GROQ_API_KEY:
    raise ValueError(" GROQ_API_KEY not found in environment variables!")
else:
    logging.info(" GROQ_API_KEY loaded successfully.")

In [52]:
print("GROQ_API_KEY:",GROQ_API_KEY)

GROQ_API_KEY: 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


###  Step 4: Load FAISS Index & Checkpoint

In [53]:
def load_index(index_path, checkpoint_path, embedding_model_name):
    logging.info("Loading embedding model...")
    embeddings = HuggingFaceEmbeddings(model_name=embedding_model_name)

    logging.info("Loading FAISS index...")
    print("index_path--->"+str(index_path))
    print("embeddings--->"+str(embeddings))
    faiss_index = FAISS.load_local(index_path, embeddings, allow_dangerous_deserialization=True)
    logging.info("FAISS index loaded successfully.")

    print("faiss_index--->"+str(faiss_index))


    if os.path.exists(checkpoint_path):
        with open(checkpoint_path, "rb") as f:
            batch = pickle.load(f)
            logging.info(f"Resuming from batch {batch}")
    else:
        batch = 0
        logging.info("No checkpoint found. Starting fresh.")

    return faiss_index, batch

###  Step 5: Tune the Retriever

In [54]:
def tune_retriever(faiss_index, k=5):
    retriever = faiss_index.as_retriever(
        search_type="mmr",
        search_kwargs={"k": k}
    )
    logging.info(f"Retriever tuned with k={k} and MMR search.")
    return retriever

###  Step 6: Create RAG Chain with Groq + LLaMA3

In [55]:
def create_rag_chain(retriever, groq_api_key):
    logging.info("Initializing LLM (LLaMA3-8b-8192)...")
    llm = ChatGroq(
        model_name="LLaMA3-8b-8192",
        groq_api_key=groq_api_key
    )

    logging.info("Creating RAG chain with RetrievalQA...")
    qa_chain = RetrievalQA.from_chain_type(
        llm=llm,
        retriever=retriever,
        return_source_documents=True,
        chain_type="stuff"
    )
    return qa_chain

###  Step 7: Run a Query

In [58]:
def run_query(chain, question):
    logging.info(f"Running query: {question}")
    result = chain({"query": question})

    print("\n Question:")
    print(question)

    print("\n Answer:")
    print(result["result"])

    print("\n Sources:")
    for doc in result["source_documents"]:
        print("- Source:", doc.metadata.get("source", "[no source]"))
        print("  Content Sample:", doc.page_content[:200], "...\n")

###  Step 8: Main Execution Block

In [57]:
# === SET PATHS ===
index_path = "."
checkpoint_path = "checkpoint_all_chunks.pkl"
embedding_model = "sentence-transformers/all-MiniLM-L6-v2"

# === LOAD & RUN ===
faiss_index, last_batch = load_index(index_path, checkpoint_path, embedding_model)
retriever = tune_retriever(faiss_index, k=5)
qa_chain = create_rag_chain(retriever, GROQ_API_KEY)

# === SAMPLE QUESTION ===
run_query(qa_chain, "What is the relationship between metabolism and the immune system?")


index_path--->.
embeddings--->client=SentenceTransformer(
  (0): Transformer({'max_seq_length': 256, 'do_lower_case': False}) with Transformer model: BertModel 
  (1): Pooling({'word_embedding_dimension': 384, 'pooling_mode_cls_token': False, 'pooling_mode_mean_tokens': True, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False, 'pooling_mode_weightedmean_tokens': False, 'pooling_mode_lasttoken': False, 'include_prompt': True})
  (2): Normalize()
) model_name='sentence-transformers/all-MiniLM-L6-v2' cache_folder=None model_kwargs={} encode_kwargs={} multi_process=False show_progress=False
faiss_index---><langchain_community.vectorstores.faiss.FAISS object at 0x79c8349f87d0>

🔍 Question:
What is the relationship between metabolism and the immune system?

✅ Answer:
According to the provided context, there is a relationship between metabolism and the immune system. The text states that "metabolism of T4 and T3 by rat hepatocytes in primary culture was measured in t